# Transformação e Qualidade dos Dados — Camada Silver

## Objetivo

Transformar os dados brutos da camada Bronze em uma tabela padronizada, com valores ausentes tratados e dados validados, preparando a base para as análises da camada Gold.

## Etapas

1. Ler os dados da tabela Bronze.
2. Padronizar os nomes das colunas e traduzir os valores categóricos.
3. Diagnosticar e tratar valores ausentes.
4. Validar nulos, duplicidades, categorias e variáveis numéricas.
5. Gravar os dados tratados como tabela Delta na camada Silver.

## Resultado esperado

Disponibilizar a tabela `projeto_titanic.silver.titanic`, com dados tratados e validados para utilização nas análises posteriores.

In [0]:
# Leitura da tabela Bronze para início do processo de transformação da camada Silver

df_silver_titanic = spark.table("projeto_titanic.bronze.titanic")

display(df_silver_titanic.limit(5))

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,null,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,null,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,null,S


In [0]:
# Padronização dos nomes das colunas para português

df_silver_titanic = df_silver_titanic.withColumnsRenamed({
    "PassengerId": "id_passageiro",
    "Survived": "sobreviveu",
    "Pclass": "classe",
    "Name": "nome",
    "Sex": "sexo",
    "Age": "idade",
    "SibSp": "irmaos_conjuges",
    "Parch": "pais_filhos",
    "Ticket": "bilhete",
    "Fare": "tarifa",
    "Cabin": "cabine",
    "Embarked": "porto_embarque"
})

display(df_silver_titanic.limit(5))

id_passageiro,sobreviveu,classe,nome,sexo,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,cabine,porto_embarque
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,null,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,null,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,null,S


In [0]:
# Tradução dos valores categóricos para facilitar a leitura e análise dos dados

from pyspark.sql.functions import *

df_silver_titanic = (
    df_silver_titanic
    .withColumn(
        "sexo",
        when(col("sexo") == "female", "feminino")
        .when(col("sexo") == "male", "masculino")
        .otherwise(col("sexo"))
    )
    .withColumn(
        "porto_embarque",
        when(col("porto_embarque") == "C", "Cherbourg")
        .when(col("porto_embarque") == "Q", "Queenstown")
        .when(col("porto_embarque") == "S", "Southampton")
        .otherwise(col("porto_embarque"))
    )
)

display(df_silver_titanic.limit(5))

id_passageiro,sobreviveu,classe,nome,sexo,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,cabine,porto_embarque
1,0,3,"Braund, Mr. Owen Harris",masculino,22.0,1,0,A/5 21171,7.25,null,Southampton
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",feminino,38.0,1,0,PC 17599,71.2833,C85,Cherbourg
3,1,3,"Heikkinen, Miss. Laina",feminino,26.0,0,0,STON/O2. 3101282,7.925,null,Southampton
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",feminino,35.0,1,0,113803,53.1,C123,Southampton
5,0,3,"Allen, Mr. William Henry",masculino,35.0,0,0,373450,8.05,null,Southampton


In [0]:
# Diagnóstico de quantidade de valores nulos na camada Silver

nulos = [
    (c, df_silver_titanic.filter(col(c).isNull()).count())
    for c in df_silver_titanic.columns
]

df_nulos = spark.createDataFrame(
    nulos,
    ["coluna", "quantidade_nulos"]
)

display(df_nulos)

coluna,quantidade_nulos
id_passageiro,0
sobreviveu,0
classe,0
nome,0
sexo,0
idade,177
irmaos_conjuges,0
pais_filhos,0
bilhete,0
tarifa,0


In [0]:
# Tratamento dos valores nulos de idade: Para preenchimento dos valores nulos de idade, foi usado a MEDIANA da idade, considerando a classe e sexo

mediana_idade = (
    df_silver_titanic
    .groupBy("classe", "sexo")
    .agg(
        percentile_approx("idade", 0.5).alias("mediana_idade")
    )
)

df_silver_titanic = (
    df_silver_titanic
    .join(mediana_idade, ["classe", "sexo"], "left")
    .withColumn(
        "idade",
        coalesce(col("idade"), col("mediana_idade"))
    )
    .drop("mediana_idade")
)

display(df_silver_titanic.limit(5))

classe,sexo,id_passageiro,sobreviveu,nome,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,cabine,porto_embarque
3,masculino,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.25,null,Southampton
1,feminino,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",38.0,1,0,PC 17599,71.2833,C85,Cherbourg
3,feminino,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.925,null,Southampton
1,feminino,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1,C123,Southampton
3,masculino,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.05,null,Southampton


In [0]:
# Tratamento da informação de cabine:
# extração da primeira letra da cabine para representar o deck.
# Para registros sem informação de cabine, é atribuída a categoria "U" (Unknown).

df_silver_titanic = (
    df_silver_titanic
    .withColumn(
        "deck",
        when(col("cabine").isNull(), lit("U"))
        .otherwise(substring(col("cabine"), 1, 1))
    )
    .drop("cabine")
)

display(df_silver_titanic.limit(5))

classe,sexo,id_passageiro,sobreviveu,nome,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,porto_embarque,deck
3,masculino,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.25,Southampton,U
1,feminino,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",38.0,1,0,PC 17599,71.2833,Cherbourg,C
3,feminino,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.925,Southampton,U
1,feminino,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1,Southampton,C
3,masculino,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.05,Southampton,U


In [0]:
# Tratamento dos valores nulos de porto_embarque: Será usado a moda para preenchimento dos nulos

moda_porto = (
    df_silver_titanic
    .select(mode("porto_embarque").alias("moda"))
    .first()["moda"]
)

df_silver_titanic = (
    df_silver_titanic
    .withColumn(
        "porto_embarque",
        when(
            col("porto_embarque").isNull(),
            lit(moda_porto)
        ).otherwise(col("porto_embarque"))
    )
)

display(df_silver_titanic.limit(5))

classe,sexo,id_passageiro,sobreviveu,nome,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,porto_embarque,deck
3,masculino,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.25,Southampton,U
1,feminino,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",38.0,1,0,PC 17599,71.2833,Cherbourg,C
3,feminino,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.925,Southampton,U
1,feminino,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1,Southampton,C
3,masculino,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.05,Southampton,U


In [0]:
# Validação dos valores nulos após os tratamentos da camada Silver

total_registros = df_silver_titanic.count()

nulos = [
    (c, total_registros, df_silver_titanic.filter(col(c).isNull()).count())
    for c in df_silver_titanic.columns
]

df_nulos = spark.createDataFrame(
    nulos,
    ["coluna", "quantidade_registros", "quantidade_nulos"]
)

display(df_nulos)

coluna,quantidade_registros,quantidade_nulos
classe,891,0
sexo,891,0
id_passageiro,891,0
sobreviveu,891,0
nome,891,0
idade,891,0
irmaos_conjuges,891,0
pais_filhos,891,0
bilhete,891,0
tarifa,891,0


In [0]:
# Validação de duplicidades e unicidade do identificador do passageiro

total_registros = df_silver_titanic.count()

passageiros_distintos = (
    df_silver_titanic
    .filter(col("id_passageiro").isNotNull())
    .select("id_passageiro")
    .distinct()
    .count()
)

registros_duplicados = (
    df_silver_titanic
    .filter(col("id_passageiro").isNotNull())
    .groupBy("id_passageiro")
    .count()
    .filter(col("count") > 1)
    .agg(sum(col("count") - 1).alias("duplicados"))
    .first()["duplicados"] or 0
)

print(f"Total de registros: {total_registros}")
print(f"Passageiros distintos: {passageiros_distintos}")
print(f"Registros duplicados: {registros_duplicados}")

Total de registros: 891
Passageiros distintos: 891
Registros duplicados: 0


In [0]:
# Validação dos valores categóricos na camada Silver

colunas_categoricas = [
    "sobreviveu",
    "classe",
    "sexo",
    "deck",
    "porto_embarque"
]

validacoes = [
    (
        c,
        df_silver_titanic
        .select(sort_array(collect_set(col(c))).alias("valores"))
        .first()["valores"]
    )
    for c in colunas_categoricas
]

df_categorias = spark.createDataFrame(
    validacoes,
    ["coluna", "valores_distintos"]
)

display(df_categorias)

coluna,valores_distintos
sobreviveu,"List(0, 1)"
classe,"List(1, 2, 3)"
sexo,"List(feminino, masculino)"
deck,"List(A, B, C, D, E, F, G, T, U)"
porto_embarque,"List(Cherbourg, Queenstown, Southampton)"


In [0]:
# Validação das variáveis numéricas na camada Silver

colunas_numericas = [
    "sobreviveu",
    "classe",
    "idade",
    "irmaos_conjuges",
    "pais_filhos",
    "tarifa"
]

validacoes = [
    (
        c,
        float(df_silver_titanic.select(min(col(c))).first()[0]),
        float(df_silver_titanic.select(max(col(c))).first()[0]),
        float(df_silver_titanic.select(avg(col(c))).first()[0]),
        df_silver_titanic.select(countDistinct(col(c))).first()[0]
    )
    for c in colunas_numericas
]

df_numericas = spark.createDataFrame(
    validacoes,
    ["coluna", "minimo", "maximo", "media", "valores_distintos"]
)

display(df_numericas)

coluna,minimo,maximo,media,valores_distintos
sobreviveu,0.0,1.0,0.3838383838383838,2
classe,1.0,3.0,2.308641975308642,3
idade,0.42,80.0,29.088855218855215,88
irmaos_conjuges,0.0,8.0,0.5230078563411896,7
pais_filhos,0.0,6.0,0.38159371492704824,7
tarifa,0.0,512.3292,32.2042079685746,248


In [0]:
# Gravação da camada Silver como tabela Delta

(
    df_silver_titanic.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.silver.titanic")
)

print("Tabela Delta gravada na camada Silver com sucesso.")

Tabela Delta gravada na camada Silver com sucesso.


In [0]:
# Visualização da Tabela Silver

display(
    spark.table("projeto_titanic.silver.titanic").limit(5)
)

classe,sexo,id_passageiro,sobreviveu,nome,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,porto_embarque,deck
3,masculino,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.25,Southampton,U
1,feminino,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",38.0,1,0,PC 17599,71.2833,Cherbourg,C
3,feminino,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.925,Southampton,U
1,feminino,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1,Southampton,C
3,masculino,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.05,Southampton,U
